# Bronze — Ingesta Kafka → Delta Lake
**LogiLake | D'LOGIA** — Capa Bronze del Medallion Architecture

Este notebook consume mensajes del topic `olist_orders` en Kafka usando
**Spark Structured Streaming** y los persiste en Delta Lake como datos raw.

Prerrequisitos:
- `pip install -r requirements.txt`
- Kafka corriendo local: `cd kafka && docker-compose up -d`
- Topic creado: `bash kafka/topic_config.sh create`
- Producer corriendo: `python kafka/olist_producer.py --data_path data/raw`

Para ingesta sin Kafka, saltar a la celda **Alternativa batch**.

In [1]:
# ── SparkSession con Delta Lake 3.1.0 + Kafka (PySpark 3.5.0) ────────────────
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("logilake_bronze")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.databricks.delta.schema.autoMerge.enabled", "true")
    # Reduce verbosidad en local
    .config("spark.ui.showConsoleProgress", "false")
)

# Incluye Kafka connector para Structured Streaming
spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=["org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"]
).getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} | Delta Lake activo")

26/04/04 01:55:03 WARN Utils: Your hostname, JuanCamiloDev resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/04 01:55:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/devcontainers/miniconda3/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/devcontainers/.ivy2/cache
The jars for the packages stored in: /home/devcontainers/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7423eec3-2f5d-43ca-b94e-99cc2233019c;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central


	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.0 in central


	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.0 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central


	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 829ms :: artifacts dl 21ms
	:: modules in use:
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	commons-logging#commons-logging;1.1.3 from central in [default]
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.commons#commons-pool2;2.11.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.3.4 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.3.4 from central in [default]
	org.apache.kafka#kafka-clients;3.4.1 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.0 from central in [default]
	org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.0 from central in [default]
	org.lz4#lz4

26/04/04 01:55:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 3.5.0 | Delta Lake activo


In [2]:
# ── Configuracion de rutas locales ────────────────────────────────────────────
import os

BASE_PATH       = "./data"
BRONZE_PATH     = f"{BASE_PATH}/bronze/olist_orders"
CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints/bronze_olist"
RAW_DATA_PATH   = f"{BASE_PATH}/raw"

# Bootstrap server de Kafka
# Opcion A - Docker en misma maquina: 'localhost:9092'
# Opcion B - Docker con host.docker.internal: 'host.docker.internal:29092'
KAFKA_BOOTSTRAP = "localhost:9092"
KAFKA_TOPIC     = "olist_orders"

os.makedirs(BRONZE_PATH, exist_ok=True)
os.makedirs(CHECKPOINT_PATH, exist_ok=True)

print(f"Bronze path:     {BRONZE_PATH}")
print(f"Checkpoint path: {CHECKPOINT_PATH}")
print(f"Kafka broker:    {KAFKA_BOOTSTRAP}")

Bronze path:     ./data/bronze/olist_orders
Checkpoint path: ./data/checkpoints/bronze_olist
Kafka broker:    localhost:9092


In [3]:
# ── Schema del payload Kafka ──────────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, FloatType, ArrayType
)

OLIST_SCHEMA = StructType([
    StructField("event_id",             StringType(),  True),
    StructField("order_id",             StringType(),  False),
    StructField("customer_id",          StringType(),  True),
    StructField("order_status",         StringType(),  True),
    StructField("order_purchase_ts",    StringType(),  True),
    StructField("order_approved_ts",    StringType(),  True),
    StructField("order_delivered_ts",   StringType(),  True),
    StructField("order_estimated_ts",   StringType(),  True),
    StructField("item_count",           IntegerType(), True),
    StructField("categories",           ArrayType(StringType()), True),
    StructField("seller_states",        ArrayType(StringType()), True),
    StructField("total_items_value",    FloatType(),   True),
    StructField("total_freight",        FloatType(),   True),
    StructField("payment_type",         StringType(),  True),
    StructField("payment_installments", IntegerType(), True),
    StructField("payment_value",        FloatType(),   True),
    StructField("review_score",         FloatType(),   True),
    StructField("ingested_at",          StringType(),  True),
    StructField("source",               StringType(),  True),
])

print("Schema definido:", len(OLIST_SCHEMA.fields), "campos")

Schema definido: 19 campos


In [4]:
# ── Leer stream desde Kafka ───────────────────────────────────────────────────
# Ejecutar solo si Kafka esta corriendo.
# Si no tienes Kafka, saltar a la celda de ingesta batch.

kafka_stream_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .option("maxOffsetsPerTrigger", 10_000)
    .option("failOnDataLoss", "false")
    .load()
)

# Deserializar JSON + metadatos Kafka
bronze_stream = (
    kafka_stream_raw
    .withColumn("value_str", F.col("value").cast("string"))
    .withColumn("data", F.from_json(F.col("value_str"), OLIST_SCHEMA))
    .select(
        "data.*",
        F.col("topic").alias("kafka_topic"),
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset"),
        F.col("timestamp").alias("kafka_timestamp"),
        F.current_timestamp().alias("bronze_ingested_at"),
    )
    .filter(F.col("order_id").isNotNull())
)

# Escribir a Delta Lake Bronze (streaming)
bronze_query = (
    bronze_stream
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(processingTime="30 seconds")
    .start(BRONZE_PATH)
)

print(f"Stream iniciado. ID: {bronze_query.id}")
# bronze_query.awaitTermination()  # Descomentar para run continuo

26/04/04 01:42:59 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Stream iniciado. ID: dc7a948f-54a3-4e10-8aba-d34c506c0d05


26/04/04 01:43:00 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/04/04 01:43:00 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node -1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
26/04/04 01:43:00 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node -1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
26/04/04 01:43:00 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node -1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.


In [6]:
# ── Monitoreo del stream ──────────────────────────────────────────────────────
import time
time.sleep(35)

print("Estado:", bronze_query.status)
if bronze_query.recentProgress:
    last = bronze_query.recentProgress[-1]
    print(f"Rows procesados: {last.get('numInputRows', 0)}")
    print(f"Throughput:      {last.get('processedRowsPerSecond', 0):.1f} rows/s")
    print(f"Batch ID:        {last.get('batchId', 'N/A')}")

# bronze_query.stop()

Estado: {'message': "Terminated with exception: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'", 'isDataAvailable': False, 'isTriggerActive': False}


In [3]:
import os, glob
from pyspark.sql import functions as F
from datetime import datetime

BASE_PATH   = "../../data"
RAW_PATH    = f"{BASE_PATH}/raw"
BRONZE_PATH = f"{BASE_PATH}/bronze"
os.makedirs(BRONZE_PATH, exist_ok=True)

# Inventario
csvs = sorted(glob.glob(f"{RAW_PATH}/*.csv"))
print(f"CSVs encontrados: {len(csvs)}")
for f in csvs:
    mb = os.path.getsize(f)/1024/1024
    print(f"  {os.path.basename(f):55s}  {mb:.2f} MB")

# Funcion de ingesta
def ingest_bronze(csv_file, table_name):
    df = (spark.read
        .option("header","true")
        .option("inferSchema","true")
        .option("encoding","UTF-8")
        .csv(f"{RAW_PATH}/{csv_file}")
        .withColumn("_bronze_ingested_at", F.current_timestamp())
        .withColumn("_source", F.lit(f"batch_csv/{table_name}"))
    )
    dest = f"{BRONZE_PATH}/{table_name}"
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema","true")
       .save(dest))
    n = spark.read.format("delta").load(dest).count()
    print(f"  OK {table_name:30s} {n:>8,} filas")
    return n

datasets = [
    ("olist_orders_dataset.csv",              "orders"),
    ("olist_order_items_dataset.csv",         "order_items"),
    ("olist_order_payments_dataset.csv",      "order_payments"),
    ("olist_order_reviews_dataset.csv",       "order_reviews"),
    ("olist_customers_dataset.csv",           "customers"),
    ("olist_sellers_dataset.csv",             "sellers"),
    ("olist_products_dataset.csv",            "products"),
    ("olist_geolocation_dataset.csv",         "geolocation"),
    ("product_category_name_translation.csv", "category_translation"),
]

print("")
print("=" * 60)
print("BRONZE INGESTION -- START")
print("=" * 60)

total = 0
for csv_file, table_name in datasets:
    try:
        total += ingest_bronze(csv_file, table_name)
    except Exception as e:
        print(f"  ERR {table_name}: {e}")

print("")
print(f"  TOTAL filas: {total:,}")
print(f"  Fin: {datetime.utcnow().isoformat()}")

CSVs encontrados: 9
  olist_customers_dataset.csv                              8.62 MB
  olist_geolocation_dataset.csv                            58.44 MB
  olist_order_items_dataset.csv                            14.72 MB
  olist_order_payments_dataset.csv                         5.51 MB
  olist_order_reviews_dataset.csv                          13.78 MB
  olist_orders_dataset.csv                                 16.84 MB
  olist_products_dataset.csv                               2.27 MB
  olist_sellers_dataset.csv                                0.17 MB
  product_category_name_translation.csv                    0.00 MB

BRONZE INGESTION -- START


26/04/04 01:55:23 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


  OK orders                           99,441 filas


  OK order_items                     112,650 filas


  OK order_payments                  103,886 filas


...

26/04/04 01:55:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers


26/04/04 01:55:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/04 01:55:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/04 01:55:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


  OK geolocation                    1,000,163 filas


  OK category_translation                 71 filas

  TOTAL filas: 1,555,860
  Fin: 2026-04-04T06:56:02.053652


/tmp/ipykernel_15184/2984853376.py:62: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print(f"  Fin: {datetime.utcnow().isoformat()}")


In [ ]:
# ── Verificacion: leer Bronze ─────────────────────────────────────────────────
bronze_df = spark.read.format("delta").load(BRONZE_PATH)

print(f"Total registros en Bronze: {bronze_df.count():,}")
bronze_df.select(
    "order_id", "order_status", "order_purchase_ts",
    "item_count", "payment_value", "review_score",
    "bronze_ingested_at"
).show(10, truncate=False)

print("\nDistribucion de estados:")
bronze_df.groupBy("order_status").count().orderBy(F.desc("count")).show()